# Crop Classification Data Analysis

This notebook performs data cleaning, preprocessing, and statistical analysis on satellite imagery data for crop classification. The dataset includes spectral band measurements (e.g., red, nir, swir16) for training and test sets. The analysis includes:
- Data cleaning and winsorization
- Statistical comparisons between train and test datasets
- Visualization of feature distributions

## Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## 1. Data Loading and Cleaning

Load the test dataset and perform initial cleaning steps:
- Remove duplicate rows
- Filter out invalid reflectance values (>1)
- Winsorize data at the 75th percentile to handle outliers

In [ ]:
# Load test dataset
t_df = pd.read_csv('/kaggle/input/new-test-amini/test (16).csv')
print(f"Original shape: {t_df.shape}")

# Remove duplicate rows
t_df.drop_duplicates(inplace=True)
print(f"Shape after dropping duplicates: {t_df.shape}")

# Define spectral band columns
band_columns = ['red', 'nir', 'swir16', 'swir22', 'blue', 'green', 'rededge1', 'rededge2', 'rededge3', 'nir08']
present_band_columns = [col for col in band_columns if col in t_df.columns]

# Remove rows with reflectance values > 1
mask_gt1 = (t_df[present_band_columns] <= 1).all(axis=1)
t_df = t_df[mask_gt1]
print(f"Shape after filtering values > 1: {t_df.shape}")

# Winsorize at 75th percentile
quartile_thresholds = t_df[present_band_columns].describe().loc['75%']
print("\nComputing 75th percentiles for each band...")
for col in present_band_columns:
    upper = quartile_thresholds[col]
    t_df[col] = np.where(t_df[col] > upper, upper, t_df[col])

# Save winsorized data
t_df.to_csv('light_test_wins continuationized.csv', index=False)
print("\nWinsorized file saved as 'light_test_winsorized.csv'")

# Display statistics after winsorization
print("\nDescriptive Statistics After Winsorization:")
print(t_df[present_band_columns].describe())

## 2. Load Train and Test Datasets

Load both training and test datasets for comparative analysis.

In [ ]:
# Load train and test datasets
train_data = pd.read_csv('/kaggle/input/new-train-amini/Training_Data_Crop_Classification.csv')
test_data = pd.read_csv('/kaggle/input/new-test-amini/test (16).csv')

## 3. Data Preview

Display the first few rows of both datasets to understand their structure.

In [ ]:
# Preview training data
train_data.head()

In [ ]:
# Preview test data
test_data.head()

## 4. Feature Distribution Analysis

Visualize the distribution of spectral band features in both train and test datasets using density plots.

In [ ]:
# Define features for analysis
features = ['blue', 'green', 'red', 'nir', 'swir16', 'swir22', 'rededge1', 'rededge2', 'rededge3', 'nir08']

# Create density plots
plt.figure(figsize=(15, 10))
for i, feature in enumerate(features, 1):
    plt.subplot(4, 3, i)
    sns.kdeplot(data=train_data[feature], label='Train', fill=True)
    sns.kdeplot(data=test_data[feature], label='Test', fill=True)
    plt.title(f'{feature} Density')
    plt.legend()
plt.tight_layout()
plt.show()

## 5. Statistical Comparison

Perform statistical comparisons between train and test datasets:
- Summary statistics
- Kolmogorov-Smirnov test for distribution differences
- Mean and variance comparisons

In [ ]:
# Summary statistics
print("Statistical Summary and KS Test Results:")
print("\nTrain Data Summary:")
print(train_data[features].describe())
print("\nTest Data Summary:")
print(test_data[features].describe())

# Kolmogorov-Smirnov test
print("\nKolmogorov-Smirnov Test (Train vs. Test):")
for feature in features:
    ks_stat, p_value = stats.ks_2samp(train_data[feature], test_data[feature])
    print(f"{feature}: KS Statistic = {ks_stat:.4f}, p-value = {p_value:.4f}")

# Mean comparison
print("\nMean Comparison (Train vs. Test):")
for feature in features:
    train_mean = train_data[feature].mean()
    test_mean = test_data[feature].mean()
    print(f"{feature}: Train Mean = {train_mean:.4f}, Test Mean = {test_mean:.4f}, Difference = {abs(train_mean - test_mean):.4f}")

# Variance comparison
print("\nVariance Comparison (Train vs. Test):")
for feature in features:
    train_var = train_data[feature].var()
    test_var = test_data[feature].var()
    print(f"{feature}: Train Variance = {train_var:.6f}, Test Variance = {test_var:.6f}")